---
## 🌡️ Section 4: Sampling Parameters

### The Parameter Map

| Parameter | Range | Effect | Best For |
|-----------|--------|--------|----------|
| `temperature` | 0.0 – 2.0 | Controls sharpness of probability distribution | Primary control |
| `top_p` | 0.0 – 1.0 | Restricts sampling to top-probability nucleus | Secondary control |
| `frequency_penalty` | 0.0 – 2.0 | Penalizes token repetition proportional to count | Long-form prose |
| `presence_penalty` | 0.0 – 2.0 | Penalizes any token already used (binary) | Topic diversity |

**Rule of thumb:**
- Structured output (JSON, code) → `temperature=0.0`
- Careful reasoning → `temperature=0.1–0.3`
- Creative prose → `temperature=0.7–1.2`

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# DEMO: Temperature sweep experiment
#
# We run the SAME prompt at 5 different temperature values.
# Each temperature is run 3 times to show intra-temperature variance.
#
# WHAT TO OBSERVE:
#   - At temperature=0.0: all 3 outputs should be identical (deterministic)
#   - At temperature=1.2: outputs should diverge significantly
#   - Find the threshold where YOUR use case first fails
# ─────────────────────────────────────────────────────────────

# This prompt represents a structured extraction task
# (highest-risk output type — feeds downstream automation)
STRUCTURED_PROMPT = """Extract the key contract terms from the text below and return ONLY valid JSON.
Format: {"vendor": "...", "sla_percent": ..., "payment_days": ..., "notice_days": ...}

Contract text:
TechCorp agrees to provide cloud infrastructure services with a 99.9% uptime SLA. 
Payment is due within 45 days of invoice. Either party may terminate with 60 days notice."""

def run_at_temperature(prompt: str, temperature: float, n_runs: int = 3) -> list:
    """
    Runs a prompt n_runs times at the given temperature.
    Returns a list of raw string outputs.
    """
    outputs = []
    for i in range(n_runs):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        outputs.append(response.choices[0].message.content.strip())
    return outputs


def check_json_validity(output: str) -> bool:
    """Returns True if the output is valid JSON, False otherwise."""
    try:
        json.loads(output)
        return True
    except json.JSONDecodeError:
        return False


# Run the sweep
temperatures = [0.0, 0.2, 0.5, 0.8, 1.2]
results = {}

print("Running temperature sweep (this may take ~30 seconds)...\n")

for temp in temperatures:
    outputs = run_at_temperature(STRUCTURED_PROMPT, temp)
    valid_count = sum(check_json_validity(o) for o in outputs)
    results[temp] = {"outputs": outputs, "compliance_rate": valid_count / len(outputs)}
    print(f"Temperature {temp:.1f}  |  JSON Valid: {valid_count}/3  |  Compliance: {valid_count/3*100:.0f}%")
    for i, out in enumerate(outputs):
        status = "OK" if check_json_validity(out) else "NOT_OK"
        # Truncate long outputs for readability
        display = out[:120] + "..." if len(out) > 120 else out
        print(f"   Run {i+1}: {status} {display}")
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────
# VISUALIZATION: Compliance rate by temperature
#
# This chart shows you exactly where your structured output
# starts failing. The compliance cliff is your tuning target.
# ─────────────────────────────────────────────────────────────

temps = list(results.keys())
compliance = [results[t]["compliance_rate"] * 100 for t in temps]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(temps, compliance, width=0.12, color=["#2ecc71" if c == 100 else "#e74c3c" if c == 0 else "#f39c12" for c in compliance])
ax.set_xlabel("Temperature", fontsize=12)
ax.set_ylabel("JSON Compliance Rate (%)", fontsize=12)
ax.set_title("Structured Output Compliance vs Temperature", fontsize=14, fontweight="bold")
ax.set_ylim(0, 110)
ax.axhline(y=100, color="gray", linestyle="--", alpha=0.5, label="100% compliance")

for bar, comp in zip(bars, compliance):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            f'{comp:.0f}%', ha='center', va='bottom', fontweight='bold')

ax.legend()
plt.tight_layout()
plt.show()

print("\n💡 For structured output tasks, use temperature at or near 0.0")
print("   The compliance cliff is where you set your upper bound.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 📝 TODO 4 — Prose vs. Structured Output (5 min)
#
# TASK:
#   Repeat the temperature sweep with a PROSE prompt
#   (e.g., a summarization task) instead of a JSON extraction task.
#
#   Define your own compliance criterion for prose:
#   - Word count within a range? (e.g., 50–100 words)
#   - Contains a required keyword?
#   - Does NOT contain a prohibited phrase?
#
#   Observe: At what temperature does prose compliance fail?
#   Compare this to the structured output cliff you found above.
#
# EXPECTED FINDING:
#   Prose tasks tolerate higher temperature than structured tasks.
#   That's why the parameter is task-specific, not global.
# ─────────────────────────────────────────────────────────────

PROSE_PROMPT = """Summarize the following in exactly 2–3 sentences. 
Do not use bullet points. Do not start with 'The document'.

Text: Transformer models process input as tokens, run attention computations across 
all positions simultaneously, and generate output autoregressively one token at a time."""

def check_prose_compliance(output: str) -> bool:
    """Define YOUR compliance criterion here."""
    # Example criteria:
    word_count = len(output.split())
    in_range = 20 <= word_count <= 80       # within word count range
    no_bullets = "•" not in output and "-" not in output[:5]  # no bullet points
    # TODO: Add your own criterion here
    return in_range and no_bullets

# --- YOUR CODE HERE ---
# Hint: Reuse run_at_temperature() with PROSE_PROMPT
# Replace check_json_validity with check_prose_compliance
pass